# 株価データ取得プログラム
このノートブックは、指定した銘柄の株価データを取得し、CSVファイルとしてダウンロードします。

## 1. 必要なライブラリのインストールと読み込み

In [ ]:
# 必要なライブラリをインストール
!pip install -q yfinance pandas matplotlib japanize-matplotlib

# ライブラリの読み込み
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
from datetime import datetime, timedelta
from google.colab import files

try:
    import japanize_matplotlib
    print('✓ 日本語フォント設定完了')
except ImportError:
    print('⚠️ japanize-matplotlib が見つかりません。ラベルが英語になる場合があります。')


## 2. 株価データ取得の設定
ここで銘柄コード、期間を設定します

In [ ]:
# ===== ここを編集してください =====

# 銘柄コード（例）
# 日本株: "7203.T" (トヨタ自動車)、"6758.T" (ソニー)
# 米国株: "AAPL" (Apple)、"MSFT" (Microsoft)、"GOOGL" (Google)
ticker_symbol = "7203.T"  # ここに取得したい銘柄コードを入力

# 期間設定（デフォルト: 直近1年間）
_today = datetime.today()
end_date   = _today.strftime("%Y-%m-%d")              # 本日
start_date = (_today - timedelta(days=365)).strftime("%Y-%m-%d")  # 1年前

# ★ 期間を手動で指定したい場合は、下の2行のコメントを外して入力してください
# start_date = "2024-01-01"
# end_date   = "2025-01-01"

# データの間隔（"1d"=日次、"1wk"=週次、"1mo"=月次）
interval = "1d"

# ===== 入力値の検証 =====
import re

date_pattern = r'^\d{4}-\d{2}-\d{2}$'
if not re.match(date_pattern, start_date) or not re.match(date_pattern, end_date):
    raise ValueError("日付は YYYY-MM-DD 形式で入力してください")

try:
    start_dt = datetime.strptime(start_date, "%Y-%m-%d")
    end_dt   = datetime.strptime(end_date,   "%Y-%m-%d")
    if start_dt >= end_dt:
        raise ValueError("開始日は終了日より前の日付にしてください")
except ValueError as e:
    raise ValueError(f"日付が無効です: {e}")

if not ticker_symbol or ticker_symbol.strip() == "":
    raise ValueError("銘柄コードを入力してください")

valid_intervals = ["1m","2m","5m","15m","30m","60m","90m","1h","1d","5d","1wk","1mo","3mo"]
if interval not in valid_intervals:
    raise ValueError(f"データ間隔は次のいずれかを指定してください: {', '.join(valid_intervals)}")

print(f"銘柄コード: {ticker_symbol}")
print(f"期間: {start_date} から {end_date}")
print(f"データ間隔: {interval}")
print("✓ 入力値の検証が完了しました")


## 3. 株価データの取得

In [ ]:
# 株価データを取得
print(f"\n{ticker_symbol} のデータを取得中...")

# yfinanceを使用してデータ取得（エラーハンドリング追加）
try:
    stock_data = yf.download(
        ticker_symbol,
        start=start_date,
        end=end_date,
        interval=interval,
        progress=True
    )
except Exception as e:
    error_msg = f"❌ データ取得中にエラーが発生しました: {str(e)}\n"
    error_msg += "以下を確認してください:\n"
    error_msg += "  - 銘柄コードが正しいか\n"
    error_msg += "  - インターネット接続が正常か\n"
    error_msg += "  - 日付範囲が適切か"
    raise RuntimeError(error_msg)

# データが取得できたか確認（重要: 空の場合は処理を中断）
if stock_data.empty:
    error_msg = "⚠️ データが取得できませんでした。\n"
    error_msg += "以下を確認してください:\n"
    error_msg += f"  - 銘柄コード「{ticker_symbol}」が正しいか\n"
    error_msg += f"  - 期間「{start_date}」から「{end_date}」にデータが存在するか\n"
    error_msg += "  - 日本株の場合は末尾に「.T」が付いているか (例: 7203.T)\n"
    error_msg += "\n⚠️ これ以降のセルは実行しないでください（空のファイルがダウンロードされます）"
    raise ValueError(error_msg)

# データ取得成功
print(f"✓ データ取得完了: {len(stock_data)} 件のデータ")
print(f"✓ カラム数: {len(stock_data.columns)}")
print(f"✓ カラム名: {', '.join(map(str, stock_data.columns))}")
print("\n最初の5件:")
print(stock_data.head())
print("\n最後の5件:")
print(stock_data.tail())

## 4. データの整形と確認

In [ ]:
# データフレームの情報を表示
print("データの概要:")
print(stock_data.info())

print("\n基本統計量:")
print(stock_data.describe())

# カラム名を日本語に変更（オプション）
# 注意: yfinanceのカラム構造が変更された場合に対応
stock_data_jp = stock_data.copy()

# 日本語化フラグの初期化
columns_converted_to_japanese = False

# カラム数の検証
expected_columns = 6
if len(stock_data_jp.columns) != expected_columns:
    print(f"⚠️ 警告: 予期しないカラム数です（期待: {expected_columns}, 実際: {len(stock_data_jp.columns)}）")
    print(f"カラム名: {list(map(str, stock_data_jp.columns))}")
    print("日本語カラム名への変換をスキップします。")
else:
    # 期待されるカラム名を検証（文字列に変換して比較）
    expected_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
    actual_cols = [str(col) for col in stock_data_jp.columns]
    
    if actual_cols == expected_cols:
        stock_data_jp.columns = ['始値', '高値', '安値', '終値', '調整後終値', '出来高']
        columns_converted_to_japanese = True  # フラグを設定
        print("✓ カラム名を日本語に変換しました")
        print("\n日本語カラム名でのプレビュー:")
        print(stock_data_jp.head())
    else:
        print(f"⚠️ 警告: カラム名が期待と異なります")
        print(f"期待: {expected_cols}")
        print(f"実際: {actual_cols}")
        print("日本語カラム名への変換をスキップします。元のカラム名を使用してください。")

## 4-2. 株価チャート（テクニカル分析付き）

取得した株価データを3つのパネルで可視化します。
- **上段**: 終値 ＋ 移動平均線（SMA20/50/200）＋ ボリンジャーバンド（±2σ）
- **中段**: 出来高（上昇日=緑、下落日=赤）
- **下段**: RSI(14) — 70超=過熱圏、30未満=売られ過ぎ圏

In [ ]:
# ----- テクニカル指標の計算 -----
_df = stock_data.copy()
if isinstance(_df.columns, pd.MultiIndex):
    _df.columns = _df.columns.get_level_values(0)

close  = _df['Close'].squeeze().astype(float)
open_  = _df['Open'].squeeze().astype(float)
volume = _df['Volume'].squeeze().astype(float)
dates  = _df.index

sma20  = close.rolling(20).mean()
sma50  = close.rolling(50).mean()
sma200 = close.rolling(200).mean()

bb_mid   = close.rolling(20).mean()
bb_std   = close.rolling(20).std()
bb_upper = bb_mid + 2 * bb_std
bb_lower = bb_mid - 2 * bb_std

_delta = close.diff()
_gain  = _delta.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
_loss  = (-_delta.clip(upper=0)).ewm(alpha=1/14, adjust=False).mean()
rsi    = 100 - 100 / (1 + _gain / _loss.replace(0, np.nan))

# 出来高の色（上昇日=緑、下落日=赤）
vol_colors = ['#26a69a' if c >= o else '#ef5350'
              for c, o in zip(close.values, open_.values)]

# ----- プロット -----
fig = plt.figure(figsize=(14, 10))
gs  = GridSpec(3, 1, figure=fig, height_ratios=[3, 1, 1], hspace=0.08)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax3 = fig.add_subplot(gs[2], sharex=ax1)

fig.suptitle(f"{ticker_symbol}  株価チャート  ({start_date} ～ {end_date})",
             fontsize=14, y=1.01)

# --- Panel 1: 終値 + SMA + ボリンジャーバンド ---
ax1.fill_between(dates, bb_lower, bb_upper, alpha=0.10, color='gray', label='BB ±2σ')
ax1.plot(dates, bb_upper, color='gray', linewidth=0.7, linestyle=':')
ax1.plot(dates, bb_lower, color='gray', linewidth=0.7, linestyle=':')
ax1.plot(dates, close,  label='終値',   color='#1565C0', linewidth=1.4, zorder=3)
if not sma20.isna().all():
    ax1.plot(dates, sma20,  label='SMA20',  color='#FF8F00', linewidth=1.1, linestyle='--')
if not sma50.isna().all():
    ax1.plot(dates, sma50,  label='SMA50',  color='#2E7D32', linewidth=1.1, linestyle='--')
if not sma200.isna().all():
    ax1.plot(dates, sma200, label='SMA200', color='#C62828', linewidth=1.1, linestyle='--')
ax1.set_ylabel('価格', fontsize=11)
ax1.legend(loc='upper left', fontsize=8, framealpha=0.7, ncol=3)
ax1.grid(True, alpha=0.25)
plt.setp(ax1.get_xticklabels(), visible=False)

# --- Panel 2: 出来高 ---
ax2.bar(dates, volume, color=vol_colors, alpha=0.80, width=0.8)
vol_ma5 = volume.rolling(5).mean()
ax2.plot(dates, vol_ma5, color='navy', linewidth=1.0, label='出来高 MA5')
ax2.set_ylabel('出来高', fontsize=11)
ax2.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f'{x/1e8:.1f}億' if x >= 1e8
                      else (f'{x/1e6:.0f}M' if x >= 1e6 else f'{x/1e3:.0f}K')))
ax2.legend(loc='upper left', fontsize=8, framealpha=0.7)
ax2.grid(True, alpha=0.25)
plt.setp(ax2.get_xticklabels(), visible=False)

# --- Panel 3: RSI ---
ax3.plot(dates, rsi, color='#6A1B9A', linewidth=1.3, label='RSI(14)')
ax3.axhline(70, color='#ef5350', linestyle='--', linewidth=1.0, alpha=0.9)
ax3.axhline(50, color='gray',    linestyle=':',  linewidth=0.8, alpha=0.7)
ax3.axhline(30, color='#26a69a', linestyle='--', linewidth=1.0, alpha=0.9)
ax3.fill_between(dates, rsi, 70, where=(rsi >= 70), alpha=0.18, color='#ef5350', label='過熱圏')
ax3.fill_between(dates, rsi, 30, where=(rsi <= 30), alpha=0.18, color='#26a69a', label='売られ過ぎ圏')
ax3.text(dates[-1], 72, '過熱(70)', color='#ef5350', fontsize=8, ha='right')
ax3.text(dates[-1], 23, '売られ過ぎ(30)', color='#26a69a', fontsize=8, ha='right')
ax3.set_ylabel('RSI(14)', fontsize=11)
ax3.set_ylim(0, 100)
ax3.grid(True, alpha=0.25)

ax3.xaxis.set_major_locator(mdates.MonthLocator())
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y/%m'))
fig.autofmt_xdate(rotation=40)

plt.tight_layout()
chart_file = f"{ticker_symbol}_chart.png"
plt.savefig(chart_file, dpi=130, bbox_inches='tight')
plt.show()
print(f"✓ チャートを保存しました: {chart_file}")


## 5. CSVファイルとしてダウンロード

In [ ]:
# ファイル名を生成
filename = f"{ticker_symbol}_{start_date}_to_{end_date}.csv"

# データの再確認（念のため）
if stock_data.empty:
    raise ValueError("❌ データが空です。このセルを実行する前にデータを取得してください。")

# CSVファイルとして保存（元のカラム名）
try:
    stock_data.to_csv(filename, encoding='utf-8-sig')  # utf-8-sigでExcelでも文字化けしない
    print(f"✓ CSVファイルを作成しました: {filename}")
    print(f"  - 行数: {len(stock_data)}")
    print(f"  - カラム数: {len(stock_data.columns)}")
except Exception as e:
    raise RuntimeError(f"❌ CSVファイルの作成に失敗しました: {str(e)}")

# ファイルをダウンロード
try:
    print("\nダウンロードを開始します...")
    files.download(filename)
    print("✓ ダウンロード完了！")
except Exception as e:
    print(f"⚠️ ダウンロードに失敗しました: {str(e)}")
    print(f"ファイル「{filename}」は作成されていますが、ダウンロードできませんでした。")

## 6. (オプション) 日本語カラム名でもダウンロード

In [ ]:
# 日本語カラム名バージョンもダウンロードしたい場合

# stock_data_jpが存在するか確認
try:
    if stock_data_jp.empty:
        raise ValueError("データが空です")
except NameError:
    raise NameError("❌ 「stock_data_jp」が定義されていません。先にセル8を実行してください。")

# 日本語化フラグの確認
try:
    is_japanese = columns_converted_to_japanese
except NameError:
    # フラグが未定義の場合、カラム名から判定
    is_japanese = '始値' in [str(col) for col in stock_data_jp.columns]

# ファイル名を動的に決定
if is_japanese:
    filename_jp = f"{ticker_symbol}_{start_date}_to_{end_date}_日本語.csv"
else:
    filename_jp = f"{ticker_symbol}_{start_date}_to_{end_date}_copy.csv"
    print("⚠️ カラムが日本語化されていないため、ファイル名を「_copy.csv」としました")

# CSVファイルとして保存
try:
    stock_data_jp.to_csv(filename_jp, encoding='utf-8-sig')
    if is_japanese:
        print(f"✓ 日本語版CSVファイルを作成しました: {filename_jp}")
    else:
        print(f"✓ CSVファイルを作成しました: {filename_jp}")
    print(f"  - 行数: {len(stock_data_jp)}")
    print(f"  - カラム数: {len(stock_data_jp.columns)}")
    print(f"  - カラム名: {', '.join(map(str, stock_data_jp.columns))}")
except Exception as e:
    raise RuntimeError(f"❌ CSVファイルの作成に失敗しました: {str(e)}")

# ファイルをダウンロード
try:
    files.download(filename_jp)
    print("✓ ダウンロード完了！")
except Exception as e:
    print(f"⚠️ ダウンロードに失敗しました: {str(e)}")
    print(f"ファイル「{filename_jp}」は作成されていますが、ダウンロードできませんでした。")

---
## 使い方のヒント

### 主要な日本株の銘柄コード例:
- トヨタ自動車: `7203.T`
- ソニーグループ: `6758.T`
- ソフトバンクグループ: `9984.T`
- キーエンス: `6861.T`
- 三菱UFJフィナンシャル・グループ: `8306.T`

### 主要な米国株の銘柄コード例:
- Apple: `AAPL`
- Microsoft: `MSFT`
- Google (Alphabet): `GOOGL`
- Amazon: `AMZN`
- Tesla: `TSLA`

### データ間隔の設定:
- `1d`: 日次データ
- `1wk`: 週次データ
- `1mo`: 月次データ
- `1h`: 時間足（直近60日間のみ）

### カラムの説明:
- **Open (始値)**: その期間の最初の取引価格
- **High (高値)**: その期間の最高価格
- **Low (安値)**: その期間の最低価格
- **Close (終値)**: その期間の最後の取引価格
- **Adj Close (調整後終値)**: 株式分割や配当を考慮した終値
- **Volume (出来高)**: その期間の取引量

---
## 7. 上昇予想銘柄のレコメンド機能

過去の株価推移（テクニカル指標）から、今後上昇が期待できる銘柄をスコアリングしてレコメンドします。

### 評価に使用するテクニカル指標
- **移動平均線 (SMA)**: 短期(20日)・中期(50日)・長期(200日)の関係から上昇トレンドを判定
- **ゴールデンクロス**: 短期SMAが中期SMAを上抜けると上昇シグナル
- **RSI (14日)**: 売られすぎ(30未満)からの反発、適度な水準(40〜60)を加点
- **モメンタム**: 直近20日・60日のリターン
- **ボリンジャーバンド**: バンド内の相対位置から過熱感を判定
- **出来高トレンド**: 直近の出来高増加で買い意欲を確認

### ⚠️ 免責事項
本機能は過去データに基づくテクニカル分析の自動化であり、将来の株価上昇を保証するものではありません。
投資判断は自己責任で行ってください。

In [ ]:
# ===== レコメンド設定 =====

# --- 対象インデックスの選択 (True = 構成銘柄を自動取得して対象に含める) ---
use_nikkei225    = True   # 日経225      （約225銘柄）
use_topix_core30 = True   # TOPIX Core30  （約30銘柄）
use_sp500        = True   # S&P 500       （約503銘柄）
use_nasdaq100    = True   # NASDAQ-100    （約100銘柄）

# --- 個別追加銘柄（インデックス外で追加したい銘柄をここに記入）---
custom_tickers = {
    # 例: "9983.T": "ファーストリテイリング",
    #     "AAPL":   "Apple",
}

# --- Stage 2（深層テクニカル分析）パラメータ ---
lookback_days   = 250   # 過去取得日数（200日SMA算出のため250推奨）
top_n           = 5     # 最終レコメンド表示件数
min_data_points = 60    # データ不足と判定する最低営業日数

# --- Stage 1（事前スクリーニング）パラメータ ---
prescreening_top_n          = 200   # Stage 1 後に深層分析へ進む最大件数
prescreening_min_20d_return = -10.0 # Stage 1 フィルタ: 20日リターン下限 (%)

print('レコメンド設定:')
print(f"  日経225       : {'有効' if use_nikkei225    else '無効'}")
print(f"  TOPIX Core30  : {'有効' if use_topix_core30 else '無効'}")
print(f"  S&P 500       : {'有効' if use_sp500        else '無効'}")
print(f"  NASDAQ-100    : {'有効' if use_nasdaq100    else '無効'}")
print(f'  カスタム銘柄  : {len(custom_tickers)} 件')
print(f'  事前スクリーニング上位 : {prescreening_top_n} 件 → 深層分析へ')
print(f'  20日リターン下限       : {prescreening_min_20d_return}%')
print(f'  最終レコメンド件数     : {top_n} 件')


### 7-2. 指数構成銘柄の自動取得

Wikipediaから各インデックスの構成銘柄を自動取得し、`candidate_tickers` を組み立てます。  
取得に失敗した場合は主要銘柄のフォールバックリストを使用します。

In [ ]:
import re as _re
from io import StringIO as _StringIO

# ============================================================
# 指数構成銘柄の取得関数（Wikipedia + フォールバック）
# ============================================================

_WIKI_HEADERS = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                   'AppleWebKit/537.36 (KHTML, like Gecko) '
                   'Chrome/120.0 Safari/537.36'),
}


def _flatten_columns(df):
    'MultiIndex（2段ヘッダー）列を _ 区切りの1段文字列に平坦化する'
    if isinstance(df.columns, pd.MultiIndex):
        df = df.copy()
        df.columns = [
            '_'.join(str(v) for v in col
                     if str(v).lower() not in ('nan', ''))
            for col in df.columns
        ]
    else:
        df = df.copy()
        df.columns = [str(c) for c in df.columns]
    return df


def _get_html_tables(url: str) -> list:
    'URLからHTMLテーブルを取得（StringIOでFutureWarning回避＋列名を平坦化）'
    try:
        import requests
        resp = requests.get(url, headers=_WIKI_HEADERS, timeout=20)
        resp.raise_for_status()
        tables = pd.read_html(_StringIO(resp.text))
    except ImportError:
        tables = pd.read_html(url)
    return [_flatten_columns(t) for t in tables]


def fetch_sp500() -> dict:
    'Wikipedia から S&P 500 構成銘柄を取得'
    FALLBACK = {
        'AAPL': 'Apple', 'MSFT': 'Microsoft', 'GOOGL': 'Alphabet',
        'AMZN': 'Amazon', 'NVDA': 'NVIDIA', 'META': 'Meta Platforms',
        'TSLA': 'Tesla', 'BRK-B': 'Berkshire Hathaway',
        'JPM': 'JPMorgan Chase', 'UNH': 'UnitedHealth',
    }
    try:
        url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
        tables = _get_html_tables(url)
        df = tables[0]
        sym_col  = next(c for c in df.columns if 'Symbol'   in str(c))
        name_col = next(c for c in df.columns if 'Security' in str(c))
        result = {}
        for _, row in df.iterrows():
            sym  = str(row[sym_col]).strip()
            name = str(row[name_col]).strip()
            if sym and sym != 'nan':
                sym = _re.sub(r'(?<=[A-Z])\.(?=[A-Z])', '-', sym)  # BRK.B -> BRK-B
                result[sym] = name
        print(f'  S&P 500      : {len(result):4d} 件取得 (Wikipedia)')
        return result
    except Exception as e:
        print(f'  [警告] S&P 500 取得失敗 ({type(e).__name__}) → フォールバック使用')
        return FALLBACK


def fetch_nasdaq100() -> dict:
    'Wikipedia から NASDAQ-100 構成銘柄を取得'
    FALLBACK = {
        'AAPL': 'Apple', 'MSFT': 'Microsoft', 'NVDA': 'NVIDIA',
        'AMZN': 'Amazon', 'META': 'Meta Platforms', 'GOOGL': 'Alphabet',
        'TSLA': 'Tesla', 'AVGO': 'Broadcom', 'COST': 'Costco', 'ASML': 'ASML',
    }
    try:
        url = 'https://en.wikipedia.org/wiki/Nasdaq-100'
        tables = _get_html_tables(url)
        df = None
        for t in tables:
            cols_lower = [str(c).lower() for c in t.columns]
            has_sym  = any('ticker' in c or 'symbol' in c for c in cols_lower)
            has_name = any('company' in c or 'name' in c for c in cols_lower)
            if has_sym and has_name and len(t) > 20:
                df = t
                break
        if df is None:
            raise ValueError('構成銘柄テーブルが見つかりません')
        sym_col  = next(c for c in df.columns
                        if 'ticker' in str(c).lower() or 'symbol' in str(c).lower())
        name_col = next(c for c in df.columns
                        if 'company' in str(c).lower() or 'name' in str(c).lower())
        result = {
            str(row[sym_col]).strip(): str(row[name_col]).strip()
            for _, row in df.iterrows()
            if str(row[sym_col]).strip() not in ('', 'nan')
        }
        print(f'  NASDAQ-100   : {len(result):4d} 件取得 (Wikipedia)')
        return result
    except Exception as e:
        print(f'  [警告] NASDAQ-100 取得失敗 ({type(e).__name__}) → フォールバック使用')
        return FALLBACK


def _jp_code_to_ticker(raw) -> 'str | None':
    '日本株コード（数値またはストリング）を 1234.T 形式に変換'
    s = str(raw).strip().replace('.T', '').replace('.t', '')
    if not s or s == 'nan':
        return None
    try:
        code = str(int(float(s)))
        return f'{code}.T' if code.isdigit() else None
    except (ValueError, OverflowError):
        return None


def _scrape_jp_index(urls, min_rows, label):
    '日本語/英語Wikipediaから コード列を持つテーブルを探して dict を返す（失敗時 None）'
    JP_CODE_KEYS = ['コード', 'code', 'ticker', 'symbol', '証券コード']
    JP_NAME_KEYS = ['銘柄', '企業', '会社', 'name', 'company']
    for url in urls:
        try:
            tables = _get_html_tables(url)
        except Exception:
            continue
        for t in tables:
            cols = [str(c) for c in t.columns]  # 取得時に平坦化済み
            cols_l = [c.lower() for c in cols]
            code_col = next((cols[i] for i, c in enumerate(cols_l)
                             if any(k.lower() in c for k in JP_CODE_KEYS)), None)
            if code_col is None or len(t) < min_rows:
                continue
            name_col = next((cols[i] for i, c in enumerate(cols_l)
                             if any(k.lower() in c for k in JP_NAME_KEYS)), None)
            if name_col is None:
                name_col = cols[1] if len(cols) > 1 else code_col
            result = {}
            for _, row in t.iterrows():
                ticker = _jp_code_to_ticker(row[code_col])
                if ticker:
                    result[ticker] = str(row[name_col]).strip()
            if len(result) >= min_rows:
                return result
    return None


NIKKEI_FALLBACK = {
        '7203.T': 'トヨタ自動車',
        '6758.T': 'ソニーグループ',
        '9984.T': 'ソフトバンクグループ',
        '6861.T': 'キーエンス',
        '8306.T': '三菱UFJフィナンシャル・グループ',
        '6098.T': 'リクルートホールディングス',
        '8035.T': '東京エレクトロン',
        '6501.T': '日立製作所',
        '7974.T': '任天堂',
        '4063.T': '信越化学工業',
        '9432.T': 'NTT',
        '8316.T': '三井住友フィナンシャルグループ',
        '8411.T': 'みずほフィナンシャルグループ',
        '4502.T': '武田薬品工業',
        '7267.T': '本田技研工業',
        '6902.T': 'デンソー',
        '6954.T': 'ファナック',
        '6594.T': 'ニデック',
        '4519.T': '中外製薬',
        '4568.T': '第一三共',
        '9433.T': 'KDDI',
        '9434.T': 'ソフトバンク',
        '6367.T': 'ダイキン工業',
        '8058.T': '三菱商事',
        '8031.T': '三井物産',
        '8001.T': '伊藤忠商事',
        '8002.T': '丸紅',
        '8053.T': '住友商事',
        '2914.T': '日本たばこ産業',
        '4661.T': 'オリエンタルランド',
        '9983.T': 'ファーストリテイリング',
        '6981.T': '村田製作所',
        '6857.T': 'アドバンテスト',
        '7741.T': 'HOYA',
        '4543.T': 'テルモ',
        '4523.T': 'エーザイ',
        '6273.T': 'SMC',
        '6645.T': 'オムロン',
        '6503.T': '三菱電機',
        '6752.T': 'パナソニック ホールディングス',
        '7751.T': 'キヤノン',
        '7733.T': 'オリンパス',
        '7011.T': '三菱重工業',
        '7012.T': '川崎重工業',
        '7013.T': 'IHI',
        '9020.T': 'JR東日本',
        '9022.T': 'JR東海',
        '9101.T': '日本郵船',
        '9104.T': '商船三井',
        '5401.T': '日本製鉄',
        '5108.T': 'ブリヂストン',
        '4452.T': '花王',
        '4911.T': '資生堂',
        '2502.T': 'アサヒグループホールディングス',
        '2503.T': 'キリンホールディングス',
        '2802.T': '味の素',
        '3382.T': 'セブン&アイ・ホールディングス',
        '8267.T': 'イオン',
        '4307.T': '野村総合研究所',
        '8604.T': '野村ホールディングス',
        '8766.T': '東京海上ホールディングス',
        '8725.T': 'MS&ADインシュアランス',
        '1605.T': 'INPEX',
        '5020.T': 'ENEOSホールディングス',
        '9501.T': '東京電力ホールディングス',
        '9531.T': '東京ガス',
        '4901.T': '富士フイルムホールディングス',
        '6701.T': 'NEC',
        '6702.T': '富士通',
        '7269.T': 'スズキ',
        '7270.T': 'SUBARU',
        '7201.T': '日産自動車',
        '3407.T': '旭化成',
        '4005.T': '住友化学',
        '4188.T': '三菱ケミカルグループ',
        '5802.T': '住友電気工業',
        '6326.T': 'クボタ',
        '7731.T': 'ニコン',
        '8801.T': '三井不動産',
        '8802.T': '三菱地所',
        '1925.T': '大和ハウス工業',
        '1928.T': '積水ハウス',
        '6146.T': 'ディスコ',
    }

TOPIX_CORE30_FALLBACK = {
        '7203.T': 'トヨタ自動車',
        '6758.T': 'ソニーグループ',
        '6861.T': 'キーエンス',
        '8306.T': '三菱UFJフィナンシャル・グループ',
        '9983.T': 'ファーストリテイリング',
        '9984.T': 'ソフトバンクグループ',
        '6098.T': 'リクルートホールディングス',
        '8035.T': '東京エレクトロン',
        '6501.T': '日立製作所',
        '9432.T': 'NTT',
        '9433.T': 'KDDI',
        '4063.T': '信越化学工業',
        '6954.T': 'ファナック',
        '4519.T': '中外製薬',
        '4568.T': '第一三共',
        '8058.T': '三菱商事',
        '8031.T': '三井物産',
        '8316.T': '三井住友フィナンシャルグループ',
        '6367.T': 'ダイキン工業',
        '6902.T': 'デンソー',
        '7267.T': '本田技研工業',
        '6981.T': '村田製作所',
        '2914.T': '日本たばこ産業',
        '4502.T': '武田薬品工業',
        '8411.T': 'みずほフィナンシャルグループ',
        '8766.T': '東京海上ホールディングス',
        '6594.T': 'ニデック',
        '7974.T': '任天堂',
        '4661.T': 'オリエンタルランド',
        '6146.T': 'ディスコ',
    }


def fetch_nikkei225() -> dict:
    '日経225 構成銘柄を取得（日本語版Wikipedia優先 → 静的リスト）'
    urls = [
        'https://ja.wikipedia.org/wiki/日経平均株価',
        'https://ja.wikipedia.org/wiki/日経225銘柄一覧',
        'https://en.wikipedia.org/wiki/Nikkei_225',
    ]
    result = _scrape_jp_index(urls, min_rows=100, label='日経225')
    if result:
        print(f'  日経225      : {len(result):4d} 件取得 (Wikipedia)')
        return result
    print(f'  [情報] 日経225 はWikipedia取得不可 → 主要 {len(NIKKEI_FALLBACK)} 銘柄の静的リストを使用')
    return dict(NIKKEI_FALLBACK)


def fetch_topix_core30() -> dict:
    'TOPIX Core30 構成銘柄を取得（日本語版Wikipedia優先 → 静的リスト）'
    urls = [
        'https://ja.wikipedia.org/wiki/TOPIX_Core30',
        'https://en.wikipedia.org/wiki/TOPIX_Core30',
    ]
    result = _scrape_jp_index(urls, min_rows=25, label='TOPIX Core30')
    if result:
        print(f'  TOPIX Core30 : {len(result):4d} 件取得 (Wikipedia)')
        return result
    print(f'  [情報] TOPIX Core30 はWikipedia取得不可 → 構成 {len(TOPIX_CORE30_FALLBACK)} 銘柄の静的リストを使用')
    return dict(TOPIX_CORE30_FALLBACK)


# ============================================================
# candidate_tickers の組み立て
# ============================================================
print('インデックス構成銘柄を取得中...\n')

candidate_tickers: dict = {}

if use_nikkei225:
    candidate_tickers.update(fetch_nikkei225())
if use_topix_core30:
    candidate_tickers.update(fetch_topix_core30())
if use_sp500:
    candidate_tickers.update(fetch_sp500())
if use_nasdaq100:
    candidate_tickers.update(fetch_nasdaq100())

# カスタム銘柄を最後にマージ（最優先）
candidate_tickers.update(custom_tickers)

print(f'\n合計候補銘柄数: {len(candidate_tickers)} 件')
print(f'（Stage 1 スクリーニング後に上位 {prescreening_top_n} 件を深層分析します）')


### 7-3. Stage 1: 事前スクリーニング（一括ダウンロード）

全候補銘柄の直近3ヶ月データを一括取得し、**20日リターン**と**出来高比**で絞り込みます。  
上位 `prescreening_top_n` 件のみを Stage 2（深層テクニカル分析）に進めます。

In [ ]:
import time

# ============================================================
# Stage 1: 一括ダウンロードによる事前スクリーニング
# ============================================================

BATCH_SIZE  = 100
all_tickers = list(candidate_tickers.keys())
batches     = [all_tickers[i:i + BATCH_SIZE] for i in range(0, len(all_tickers), BATCH_SIZE)]

print('Stage 1 事前スクリーニング開始')
print(f'  対象銘柄数 : {len(all_tickers):5d} 件')
print(f'  バッチ数   : {len(batches):5d} バッチ（各最大 {BATCH_SIZE} 件）')
print(f'  取得期間   : 直近 3 ヶ月\n')

screening_records = []
stage1_start = time.time()

for batch_idx, batch in enumerate(batches):
    print(f'  [{batch_idx + 1:2d}/{len(batches)}] {len(batch):3d} 件をダウンロード中...', end=' ', flush=True)
    try:
        bulk_df = yf.download(
            batch,
            period='3mo',
            group_by='ticker',
            progress=False,
            auto_adjust=False,
        )
    except Exception as e:
        print(f'[エラー: {type(e).__name__}] — スキップ')
        continue

    if bulk_df is None or bulk_df.empty:
        print('[データなし] — スキップ')
        continue

    is_multi   = isinstance(bulk_df.columns, pd.MultiIndex)
    lv0        = set(bulk_df.columns.get_level_values(0).tolist()) if is_multi else set()
    field_first = ('Close' in lv0) if is_multi else True  # True: (field, ticker)

    batch_ok = 0
    for ticker in batch:
        try:
            if is_multi:
                if field_first:
                    if 'Close' not in bulk_df or ticker not in bulk_df['Close'].columns:
                        continue
                    close_s  = bulk_df['Close'][ticker].dropna()
                    volume_s = bulk_df['Volume'][ticker].dropna()
                else:
                    if ticker not in lv0:
                        continue
                    close_s  = bulk_df[ticker]['Close'].dropna()
                    volume_s = bulk_df[ticker]['Volume'].dropna()
            else:
                close_s  = bulk_df['Close'].dropna()
                volume_s = bulk_df['Volume'].dropna()

            close_s  = close_s.astype(float)
            volume_s = volume_s.astype(float)

            if len(close_s) < 21:
                continue

            ret_20d   = (float(close_s.iloc[-1]) / float(close_s.iloc[-21]) - 1) * 100
            vol_5d    = float(volume_s.tail(5).mean())
            vol_20d   = float(volume_s.tail(20).mean())
            vol_ratio = vol_5d / vol_20d if vol_20d > 0 else 0.0

            screening_records.append({
                'ticker'    : ticker,
                'name'      : candidate_tickers.get(ticker, ticker),
                'return_20d': ret_20d,
                'vol_ratio' : vol_ratio,
            })
            batch_ok += 1
        except Exception:
            continue

    print(f'OK ({batch_ok}/{len(batch)} 件)')

stage1_elapsed = time.time() - stage1_start

if not screening_records:
    raise RuntimeError('❌ Stage 1 でデータが取得できませんでした。ネットワーク接続を確認してください。')

screen_df = pd.DataFrame(screening_records)

filtered_df = screen_df[
    (screen_df['return_20d'] >= prescreening_min_20d_return) &
    (screen_df['vol_ratio']  >  0)
].copy()

filtered_df = filtered_df.sort_values('return_20d', ascending=False).head(prescreening_top_n)

prescreened_tickers = {
    row['ticker']: row['name']
    for _, row in filtered_df.iterrows()
}

print()
print('=' * 65)
print(f'Stage 1 完了  （{stage1_elapsed:.1f} 秒）')
print('=' * 65)
print(f'  データ取得成功 : {len(screen_df):4d} / {len(all_tickers)} 件')
print(f'  フィルタ通過   : {len(filtered_df):4d} 件  （20日リターン >= {prescreening_min_20d_return}%）')
print(f'  Stage 2 対象   : {len(prescreened_tickers):4d} 件')
print()
print('  スクリーニング上位 10 件:')
for i, (_, row) in enumerate(filtered_df.head(10).iterrows(), 1):
    print(f"    {i:2d}. {row['ticker']:12s}  {row['name'][:22]:22s}  "
          f"20日: {row['return_20d']:+6.1f}%  出来高比: {row['vol_ratio']:.2f}")


In [ ]:
# ===== テクニカル指標の計算とスコアリング =====
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta


def compute_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    """RSI（相対力指数）を計算"""
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi


def analyze_ticker(ticker: str, name: str, period_days: int) -> dict | None:
    """1銘柄分のテクニカル指標とスコアを算出する。データ不足ならNoneを返す。"""
    end = datetime.today()
    start = end - timedelta(days=period_days + 30)  # 余裕を持って取得

    try:
        df = yf.download(ticker, start=start.strftime("%Y-%m-%d"),
                         end=end.strftime("%Y-%m-%d"), progress=False, auto_adjust=False)
    except Exception as e:
        print(f"  [スキップ] {ticker}: 取得エラー ({e})")
        return None

    if df.empty or len(df) < min_data_points:
        print(f"  [スキップ] {ticker}: データ不足 ({len(df) if not df.empty else 0} 日)")
        return None

    # MultiIndexカラム対応（yfinanceのバージョン差を吸収）
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    close = df["Close"].astype(float)
    volume = df["Volume"].astype(float)

    # 各種指標
    sma20 = close.rolling(20).mean()
    sma50 = close.rolling(50).mean()
    sma200 = close.rolling(min(200, len(close))).mean()
    rsi14 = compute_rsi(close, 14)
    bb_mid = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    bb_upper = bb_mid + 2 * bb_std
    bb_lower = bb_mid - 2 * bb_std

    last_close = float(close.iloc[-1])
    last_sma20 = float(sma20.iloc[-1]) if not np.isnan(sma20.iloc[-1]) else np.nan
    last_sma50 = float(sma50.iloc[-1]) if not np.isnan(sma50.iloc[-1]) else np.nan
    last_sma200 = float(sma200.iloc[-1]) if not np.isnan(sma200.iloc[-1]) else np.nan
    last_rsi = float(rsi14.iloc[-1]) if not np.isnan(rsi14.iloc[-1]) else np.nan

    # モメンタム（リターン）
    ret_20d = (last_close / float(close.iloc[-21]) - 1) * 100 if len(close) > 21 else np.nan
    ret_60d = (last_close / float(close.iloc[-61]) - 1) * 100 if len(close) > 61 else np.nan

    # 出来高トレンド: 直近5日平均 vs 過去20日平均
    vol_recent = float(volume.tail(5).mean())
    vol_base = float(volume.tail(20).mean())
    vol_ratio = vol_recent / vol_base if vol_base > 0 else 1.0

    # ボリンジャー位置（0=下限, 1=上限）
    bb_pos = np.nan
    if not np.isnan(bb_upper.iloc[-1]) and not np.isnan(bb_lower.iloc[-1]):
        rng = float(bb_upper.iloc[-1]) - float(bb_lower.iloc[-1])
        if rng > 0:
            bb_pos = (last_close - float(bb_lower.iloc[-1])) / rng

    # ===== スコアリング（合計 100点満点）=====
    score = 0.0
    reasons = []

    # 1. トレンド: 終値 > SMA20 > SMA50 > SMA200 (各10点, 最大30点)
    if not np.isnan(last_sma20) and last_close > last_sma20:
        score += 10
        reasons.append("終値>SMA20")
    if not np.isnan(last_sma50) and not np.isnan(last_sma20) and last_sma20 > last_sma50:
        score += 10
        reasons.append("SMA20>SMA50")
    if not np.isnan(last_sma200) and not np.isnan(last_sma50) and last_sma50 > last_sma200:
        score += 10
        reasons.append("SMA50>SMA200")

    # 2. ゴールデンクロス（直近10営業日以内）: 15点
    if len(sma20.dropna()) >= 11 and len(sma50.dropna()) >= 11:
        recent20 = sma20.tail(11).values
        recent50 = sma50.tail(11).values
        crossed = False
        for i in range(1, len(recent20)):
            if recent20[i - 1] <= recent50[i - 1] and recent20[i] > recent50[i]:
                crossed = True
                break
        if crossed:
            score += 15
            reasons.append("直近ゴールデンクロス")

    # 3. RSI: 売られすぎ反発(30〜45)で+15点、適度(45〜65)で+10点、過熱(>75)は減点
    if not np.isnan(last_rsi):
        if 30 <= last_rsi < 45:
            score += 15
            reasons.append(f"RSI反発圏({last_rsi:.0f})")
        elif 45 <= last_rsi <= 65:
            score += 10
            reasons.append(f"RSI健全({last_rsi:.0f})")
        elif last_rsi > 75:
            score -= 10
            reasons.append(f"RSI過熱({last_rsi:.0f})")

    # 4. モメンタム: 20日リターン(最大15点)、60日リターン(最大10点)
    if not np.isnan(ret_20d):
        score += max(min(ret_20d, 15), -10) * (15 / 15)  # 簡易マッピング
        if ret_20d > 0:
            reasons.append(f"20日+{ret_20d:.1f}%")
    if not np.isnan(ret_60d):
        score += max(min(ret_60d / 2, 10), -5)
        if ret_60d > 0:
            reasons.append(f"60日+{ret_60d:.1f}%")

    # 5. 出来高トレンド: 直近5日平均が20日平均より高ければ+5、1.5倍以上で+10
    if vol_ratio >= 1.5:
        score += 10
        reasons.append(f"出来高急増(x{vol_ratio:.1f})")
    elif vol_ratio >= 1.1:
        score += 5
        reasons.append(f"出来高増(x{vol_ratio:.2f})")

    # 6. ボリンジャーバンド: 中央付近(0.3〜0.7)で安定上昇なら+5、上限超え(>1.0)は減点
    if not np.isnan(bb_pos):
        if 0.3 <= bb_pos <= 0.7:
            score += 5
        elif bb_pos > 1.0:
            score -= 5
            reasons.append("BB上抜け(過熱)")

    # ===== エントリースコア（今が買い時かを判定）=====
    # 前提: 終値 >= SMA200（マクロ上昇トレンド内であること）
    entry_s = 0
    entry_r = []
    above_sma200 = (not np.isnan(last_sma200)) and (last_close >= last_sma200)
    if above_sma200:
        # 1. RSI回復ゾーン (30〜55): 売られすぎからの回復・健全な水準
        if not np.isnan(last_rsi) and 30 <= last_rsi <= 55:
            entry_s += 30
            entry_r.append(f'RSI回復({last_rsi:.0f})')
        # 2. 価格がSMA20を直近5日で下から上抜け（反転シグナル）
        if (not np.isnan(last_sma20) and last_close > last_sma20
                and len(close) >= 6 and len(sma20.dropna()) >= 6):
            win = [(float(close.iloc[i]), float(sma20.iloc[i]))
                   for i in range(-6, -1) if not np.isnan(sma20.iloc[i])]
            if win and any(c < s for c, s in win):
                entry_s += 25
                entry_r.append('SMA20上抜け')
        # 3. ゴールデンクロス直近20日（エントリー窓を広げて検出）
        if len(sma20.dropna()) >= 21 and len(sma50.dropna()) >= 21:
            r20e = sma20.tail(21).values
            r50e = sma50.tail(21).values
            if any(r20e[i-1] <= r50e[i-1] and r20e[i] > r50e[i]
                   for i in range(1, len(r20e))):
                entry_s += 20
                entry_r.append('GC直近20日')
        # 4. 価格がBBミッド以下（まだ上昇余地あり）
        bb_mid_last = float(bb_mid.iloc[-1]) if not np.isnan(bb_mid.iloc[-1]) else np.nan
        if not np.isnan(bb_mid_last) and last_close < bb_mid_last:
            entry_s += 15
            entry_r.append('BB下半分')
        # 5. 出来高急増（直近5日 > 20日平均 × 1.2）
        if vol_ratio > 1.2:
            entry_s += 15
            entry_r.append(f'出来高急増(x{vol_ratio:.1f})')


    return {
        "ticker": ticker,
        "name": name,
        "終値": round(last_close, 2),
        "SMA20": round(last_sma20, 2) if not np.isnan(last_sma20) else None,
        "SMA50": round(last_sma50, 2) if not np.isnan(last_sma50) else None,
        "SMA200": round(last_sma200, 2) if not np.isnan(last_sma200) else None,
        "RSI14": round(last_rsi, 1) if not np.isnan(last_rsi) else None,
        "20日リターン%": round(ret_20d, 2) if not np.isnan(ret_20d) else None,
        "60日リターン%": round(ret_60d, 2) if not np.isnan(ret_60d) else None,
        "出来高比": round(vol_ratio, 2),
        "BB位置": round(bb_pos, 2) if not np.isnan(bb_pos) else None,
        "スコア": round(score, 1),
        "シグナル": " / ".join(reasons) if reasons else "-",
        "entry_score": round(entry_s, 1),
        "entry_reasons": " / ".join(entry_r) if entry_r else "-",
    }


print("テクニカル分析の関数を定義しました。")


In [ ]:
# ===== Stage 2: 深層テクニカル分析 & 上昇期待度ランキング =====
import time as _time2

stage2_start  = _time2.time()
total_stage2  = len(prescreened_tickers)
print(f'Stage 2 深層分析: {total_stage2} 銘柄 × 直近 {lookback_days} 日分を分析中...\n')

results = []
for i, (tkr, nm) in enumerate(prescreened_tickers.items(), 1):
    pct = i / total_stage2 * 100
    print(f'  [{i:3d}/{total_stage2}] {pct:4.0f}%  {tkr} ({nm})')
    r = analyze_ticker(tkr, nm, lookback_days)
    if r is not None:
        results.append(r)

stage2_elapsed = _time2.time() - stage2_start
try:
    _total_str = f' / 合計 {stage1_elapsed + stage2_elapsed:.0f} 秒'
except NameError:
    _total_str = ''
print(f'\nStage 2 完了  （{stage2_elapsed:.1f} 秒{_total_str}）\n')

if not results:
    raise RuntimeError('❌ 分析可能なデータが取得できませんでした。銘柄リストやネットワークを確認してください。')

# DataFrameに集約し、スコアで降順ソート
ranking_df = pd.DataFrame(results).sort_values("スコア", ascending=False).reset_index(drop=True)
ranking_df.index = ranking_df.index + 1  # 1始まり
ranking_df.index.name = "順位"

print("\n" + "=" * 70)
print(f"📈 上昇期待度ランキング TOP {top_n}")
print("=" * 70)

top_df = ranking_df.head(top_n)
for rank, row in top_df.iterrows():
    print(f"\n第{rank}位: {row['ticker']} ({row['name']})  スコア: {row['スコア']}")
    print(f"  終値 {row['終値']} / RSI {row['RSI14']} / 20日 {row['20日リターン%']}% / 60日 {row['60日リターン%']}%")
    print(f"  シグナル: {row['シグナル']}")

print("\n" + "=" * 70)
print("全銘柄のスコア表:")
print("=" * 70)
display_cols = ["ticker", "name", "スコア", "終値", "RSI14", "20日リターン%", "60日リターン%", "出来高比", "シグナル"]
print(ranking_df[display_cols].to_string())

# ----- 狙い目銘柄ランキング（今が買い時エントリー評価）-----
_entry_list = [r for r in results if r.get('entry_score', 0) > 0]
if _entry_list:
    entry_df = (pd.DataFrame(_entry_list)
                .sort_values('entry_score', ascending=False)
                .reset_index(drop=True))
    entry_df.index += 1
    print('\n' + '=' * 70)
    _eh = min(5, len(entry_df))
    print(f'🎯 狙い目銘柄 TOP {_eh}  （上昇の入口にいると判定された銘柄）')
    print('=' * 70)
    print('  ※ テクニカル上位とは別に「今が買い時」という観点でスクリーニング')
    print('  ※ RSI回復 / SMA20上抜け / GC直後 / BB下半分 / 出来高増 を評価\n')
    for idx, row in entry_df.head(5).iterrows():
        print(f"  #{idx}  {row['ticker']} ({row['name']})")
        print(f"      狙い目スコア: {row['entry_score']}  テクニカルスコア: {row['スコア']}")
        print(f"      エントリー理由: {row['entry_reasons']}")
else:
    entry_df = pd.DataFrame()
    print('\n🎯 狙い目銘柄: 現時点でエントリー条件を満たす銘柄がありません')

# CSV出力
recommend_filename = f"recommend_top_{datetime.today().strftime('%Y%m%d')}.csv"
try:
    ranking_df.to_csv(recommend_filename, encoding="utf-8-sig")
    print(f"\n✓ ランキングをCSVに保存しました: {recommend_filename}")
    try:
        from google.colab import files as colab_files
        colab_files.download(recommend_filename)
    except Exception:
        print("  (Colab以外の環境ではダウンロードはスキップされます)")
except Exception as e:
    print(f"⚠️ CSV保存に失敗: {e}")

print("\n⚠️ 免責事項: このスコアは過去データからのテクニカル分析であり、将来の値上がりを保証するものではありません。")
print("   投資判断はご自身の責任で行ってください。")

# ----- スコアのバーチャート -----
fig, axes2 = plt.subplots(1, 2, figsize=(16, max(4, len(ranking_df) * 0.5 + 1)))

# 左: スコアランキング（横棒グラフ）
ax_bar = axes2[0]
labels = [f"{row['ticker']} ({row['name']})" for _, row in ranking_df.iterrows()]
scores_list = ranking_df['スコア'].tolist()
bar_colors = ['#1565C0' if i < top_n else '#B0BEC5' for i in range(len(ranking_df))]
hbars = ax_bar.barh(labels, scores_list, color=bar_colors, edgecolor='white', height=0.65)
for bar, sc in zip(hbars, scores_list):
    ax_bar.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                f'{sc:.1f}', va='center', fontsize=9)
ax_bar.set_xlabel('スコア', fontsize=11)
ax_bar.set_title('上昇期待度スコア ランキング', fontsize=12)
ax_bar.invert_yaxis()
ax_bar.set_xlim(0, max(scores_list) * 1.18 if max(scores_list) > 0 else 10)
ax_bar.axvline(50, color='orange', linestyle='--', linewidth=0.9, alpha=0.7)
ax_bar.grid(axis='x', alpha=0.3)

# 右: 上位銘柄のテクニカル指標ヒートマップ
ax_heat = axes2[1]
heat_cols = ['RSI14', '20日リターン%', '60日リターン%', '出来高比']
heat_labels = ['RSI14', '20日\nリターン%', '60日\nリターン%', '出来高比']
top_data = top_df[heat_cols].copy()
top_data.index = [row['ticker'] for _, row in top_df.iterrows()]
top_data = top_data.astype(float).fillna(0)
_norm = top_data.copy()
for col in _norm.columns:
    mn, mx = _norm[col].min(), _norm[col].max()
    _norm[col] = (_norm[col] - mn) / (mx - mn + 1e-9)
im = ax_heat.imshow(_norm.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax_heat.set_xticks(range(len(heat_labels)))
ax_heat.set_xticklabels(heat_labels, fontsize=9)
ax_heat.set_yticks(range(len(top_data.index)))
ax_heat.set_yticklabels(top_data.index, fontsize=9)
ax_heat.set_title(f'TOP {top_n} テクニカル指標ヒートマップ', fontsize=12)
for i in range(len(top_data.index)):
    for j in range(len(heat_cols)):
        val = top_data.iloc[i, j]
        nv = _norm.iloc[i, j]
        ax_heat.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=9,
                     color='black' if 0.2 < nv < 0.8 else 'white')
plt.colorbar(im, ax=ax_heat, shrink=0.8, label='相対スコア')

plt.tight_layout()
plt.savefig('recommend_chart.png', dpi=130, bbox_inches='tight')
plt.show()
print("\n✓ レコメンドチャートを保存しました: recommend_chart.png")


---
## 8. AI予測モデルによる将来株価予測（Stage 3）

Stage 2 で抽出した上位銘柄に対し、**3種類のニューラルネットワーク**で訓練し、
**最適モデルを自動選択**して将来の株価を予測します。

### 比較する3モデル
| モデル | 構造 | 特徴 |
|---|---|---|
| **LSTM-Simple**  | LSTM(32) + Dense(1) | 軽量・高速 |
| **LSTM-Stacked** | LSTM(50) × 3 + Dropout + Dense(1) | 表現力高め |
| **GRU**          | GRU(50) × 2 + Dropout + Dense(1) | LSTM の軽量代替 |

### 評価指標と読み方
| 指標 | 意味 | 目安 |
|---|---|---|
| **MAPE (%)** | 価格予測の誤差率 — 小さいほど精度が高い | 日本株で 2〜5% が目安 |
| **方向精度 (Dir Acc)** | 翌日「上がるか下がるか」の正解率 | 50%=ランダム、55%以上で有意 |
| **AI予測リターン** | 最適モデルによる N 日後の予測変化率 | MAPE が低い銘柄の値が信頼できる |

### 🔑 有望銘柄の見方（複合シグナル）
テクニカルスコアと AI 予測の**両方が上昇を示す銘柄**が最も強いシグナルです。
最後のセルに **「複合スコアランキング」** を表示します。

> **技術的補足**: 従来の価格ベース LSTM は「平均回帰バイアス」が生じやすいため、
> 本実装では**対数収益率**を学習対象に変更しています。これにより収益率の定常性を
> 利用でき、急騰後のモデルが一方的にマイナスを予測するバイアスを軽減します。

### ⚠️ 処理時間と注意事項
- Colab CPU で銘柄あたり 1〜3 分目安（GPU で大幅短縮）
- 本予測は教育・学習目的です。投資判断は自己責任で行ってください

In [ ]:
# ===== AI予測（Stage 3）パラメータ =====

RUN_NN_PREDICTION  = True   # False にするとAI予測をスキップ
NN_TOP_K           = 3      # Stage 2 上位 何銘柄に対して予測を実行するか
NN_LOOK_BACK       = 60     # 入力シーケンス長（過去何日の対数収益率で予測するか）
NN_TRAIN_YEARS     = 1.5    # 訓練に使う過去年数（1.5年 = 直近データ重視）
NN_EPOCHS          = 30     # 訓練エポック数（EarlyStopping=5 で早期終了）
NN_BATCH           = 32     # バッチサイズ
NN_FORECAST_DAYS   = 30     # 何日先まで予測するか（反復予測）

if RUN_NN_PREDICTION:
    print('AI予測（Stage 3）設定:')
    print(f'  対象銘柄数   : Stage 2 上位 {NN_TOP_K} 銘柄')
    print(f'  訓練期間     : 過去 {NN_TRAIN_YEARS} 年分（直近データ重視）')
    print(f'  入力         : {NN_LOOK_BACK} 日間の対数収益率')
    print(f'  予測         : {NN_FORECAST_DAYS} 日先の株価')
    print(f'  訓練設定     : {NN_EPOCHS} エポック, EarlyStopping(patience=5)')
else:
    print('AI予測（Stage 3）は無効化されています。RUN_NN_PREDICTION = True で有効化できます。')


# --- 狙い目ハントモード ---
NN_HUNT_MODE = True   # True: 「今が買い時」エントリー候補もAIで追加分析
NN_HUNT_K    = 5      # 狙い目候補から何銘柄をAI分析するか
if RUN_NN_PREDICTION and NN_HUNT_MODE:
    print(f'  狙い目ハント : 有効（エントリー候補 上位 {NN_HUNT_K} 銘柄を追加）')


In [ ]:
# ===== モデル定義と学習・評価関数（対数収益率ベース）=====
if RUN_NN_PREDICTION:
    import os
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping

    print(f'✓ TensorFlow {tf.__version__} 読み込み完了')

    # ----- 3種類のモデル定義（アーキテクチャ変更なし）-----
    def build_lstm_simple(look_back):
        m = Sequential([
            LSTM(32, input_shape=(look_back, 1)),
            Dropout(0.2), Dense(1),
        ])
        m.compile(optimizer='adam', loss='mse')
        return m

    def build_lstm_stacked(look_back):
        m = Sequential([
            LSTM(50, return_sequences=True, input_shape=(look_back, 1)),
            Dropout(0.2),
            LSTM(50, return_sequences=True), Dropout(0.2),
            LSTM(50), Dropout(0.2), Dense(1),
        ])
        m.compile(optimizer='adam', loss='mse')
        return m

    def build_gru(look_back):
        m = Sequential([
            GRU(50, return_sequences=True, input_shape=(look_back, 1)),
            Dropout(0.2), GRU(50), Dropout(0.2), Dense(1),
        ])
        m.compile(optimizer='adam', loss='mse')
        return m

    MODEL_BUILDERS = {
        'LSTM-Simple':  build_lstm_simple,
        'LSTM-Stacked': build_lstm_stacked,
        'GRU':          build_gru,
    }

    # ----- シーケンス生成（対数収益率配列用）-----
    def make_sequences(arr2d, look_back):
        'shape (N,1) → X:(N-look_back, look_back, 1), y:(N-look_back,)'
        X, y = [], []
        for i in range(look_back, len(arr2d)):
            X.append(arr2d[i - look_back:i, 0])
            y.append(arr2d[i, 0])
        return np.array(X).reshape(-1, look_back, 1), np.array(y)

    # ----- 1モデルの訓練・評価（対数収益率ベース）-----
    def train_and_evaluate_logrtn(builder_fn, X_tr, y_tr, X_te, y_te,
                                   log_ret_test, start_price_test,
                                   close_test, ret_scale, epochs, batch):
        'Returns (model, pred_prices, mape, dir_acc, composite)'
        tf.random.set_seed(42)
        model = builder_fn(X_tr.shape[1])
        es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
        model.fit(X_tr, y_tr, epochs=epochs, batch_size=batch,
                  validation_data=(X_te, y_te), callbacks=[es], verbose=0)

        pred_scaled = model.predict(X_te, verbose=0).flatten()
        pred_log_ret = pred_scaled * ret_scale          # unscale to actual log-return
        pred_prices  = start_price_test * np.exp(np.cumsum(pred_log_ret))

        n = min(len(close_test), len(pred_prices))
        actual = close_test[:n]
        pred   = pred_prices[:n]

        mape    = float(np.mean(np.abs((actual - pred) / (actual + 1e-10))) * 100)
        dir_act = (log_ret_test[:n] > 0).astype(int)
        dir_prd = (pred_log_ret[:n] > 0).astype(int)
        dir_acc = float(np.mean(dir_act == dir_prd))

        # 複合指標: MAPEが低く、方向精度が高いほど小さい（最小化目標）
        composite = mape / (dir_acc + 0.01)

        return model, pred_prices, mape, dir_acc, composite

    # ----- 1銘柄の全工程: 取得→前処理→訓練→最適選択→将来予測 -----
    def predict_with_best_model(ticker, name, train_years, look_back,
                                 forecast_days, epochs, batch):
        'log-return ベースで3モデルを比較し、最適モデルで将来予測する'
        from datetime import datetime as _dt, timedelta as _td
        end   = _dt.today()
        start = end - _td(days=int(train_years * 365) + 60)
        try:
            df = yf.download(ticker, start=start.strftime('%Y-%m-%d'),
                             end=end.strftime('%Y-%m-%d'),
                             progress=False, auto_adjust=False)
        except Exception:
            return None
        if df is None or df.empty:
            return None
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        if 'Close' not in df.columns or len(df) < look_back + 80:
            return None

        close_arr = df['Close'].astype(float).values

        # --- 対数収益率の計算とスケーリング ---
        log_ret = np.diff(np.log(close_arr))   # shape: (N-1,)
        N = len(log_ret)
        train_size = int(N * 0.8)
        if train_size < look_back + 20:
            return None

        ret_std   = float(np.std(log_ret[:train_size])) + 1e-8
        ret_scale = 3.0 * ret_std             # 正規化係数（±3σ を ±1 にマッピング）
        scaled    = (log_ret / ret_scale).reshape(-1, 1)  # shape (N-1, 1)

        X_train, y_train = make_sequences(scaled[:train_size], look_back)
        X_test,  y_test  = make_sequences(scaled[train_size - look_back:], look_back)

        # テスト期間の実績価格（予測結果との比較用）
        start_price_test = float(close_arr[train_size])
        close_test       = close_arr[train_size + 1:]   # shape: (N-train_size,)
        log_ret_test     = log_ret[train_size:]          # shape: (N-train_size,)

        # --- 全モデルを訓練・比較 ---
        per_model = {}
        best = {'name': None, 'comp': float('inf'),
                'mape': None, 'dir_acc': None, 'model': None, 'pred': None}

        for mname, builder in MODEL_BUILDERS.items():
            try:
                m, pred_p, mape, dir_acc, comp = train_and_evaluate_logrtn(
                    builder, X_train, y_train, X_test, y_test,
                    log_ret_test, start_price_test, close_test,
                    ret_scale, epochs, batch)
                per_model[mname] = {'mape': mape, 'dir_acc': dir_acc, 'composite': comp}
                if comp < best['comp']:
                    best = {'name': mname, 'comp': comp, 'mape': mape,
                            'dir_acc': dir_acc, 'model': m, 'pred': pred_p}
            except Exception as e:
                per_model[mname] = {'mape': None, 'dir_acc': None, 'composite': None,
                                    'error': str(e)}

        if best['model'] is None:
            return None

        # --- 最適モデルで将来 forecast_days 日を反復予測 ---
        seq = scaled[-look_back:].reshape(1, look_back, 1).copy()
        forecast_log_rets = []
        for _ in range(forecast_days):
            nxt = best['model'].predict(seq, verbose=0)
            raw_ret = float(nxt[0, 0]) * ret_scale      # 実際の対数収益率
            forecast_log_rets.append(raw_ret)
            seq = np.append(seq[:, 1:, :], nxt.reshape(1, 1, 1), axis=1)

        forecast_prices = float(close_arr[-1]) * np.exp(np.cumsum(forecast_log_rets))
        current_price   = float(close_arr[-1])
        forecast_end    = float(forecast_prices[-1])
        forecast_ret    = (forecast_end / current_price - 1) * 100

        # AI信頼スコア: 方向が強く、MAPEが低いほど高い
        # tanh で ±1 に正規化した方向性 × 信頼度 (1/MAPE)
        ai_signal    = float(np.tanh(forecast_ret / 5.0))   # -1〜+1
        ai_confidence = 1.0 / (best['mape'] + 1.0)           # MAPEが低いほど高い
        ai_score      = ai_signal * ai_confidence * 100       # スコール換算

        n_pred = min(len(close_test), len(best['pred']))
        test_dates = df.index[train_size + 1 : train_size + 1 + n_pred]

        return {
            'ticker': ticker, 'name': name,
            'best_model': best['name'],
            'best_mape':  best['mape'],
            'best_dir_acc': best['dir_acc'],
            'best_composite': best['comp'],
            'per_model': per_model,
            'current_price': current_price,
            'forecast_end_price': forecast_end,
            'forecast_return': forecast_ret,
            'ai_signal':    ai_signal,
            'ai_confidence': ai_confidence,
            'ai_score':     ai_score,
            'forecast_dates':  pd.date_range(df.index[-1] + pd.Timedelta(days=1),
                                              periods=forecast_days),
            'forecast_prices': forecast_prices,
            'history_dates':   df.index,
            'history_prices':  close_arr,
            'test_dates':      test_dates,
            'test_actual':     close_test[:n_pred],
            'test_pred':       best['pred'][:n_pred],
        }

    print('✓ モデル定義完了（対数収益率ベース学習 / LSTM-Simple / LSTM-Stacked / GRU）')


In [ ]:
# ===== Stage 3 実行: テクニカル上位 + 狙い目候補のAI予測 =====
if RUN_NN_PREDICTION:
    stage3_start = time.time()

    # テクニカル上位銘柄
    tech_targets = [(row['ticker'], row['name'], 'テクニカル上位')
                    for _, row in ranking_df.head(NN_TOP_K).iterrows()]

    # 狙い目ハント候補（テクニカル上位と重複しないもの）
    hunt_targets = []
    if NN_HUNT_MODE and not entry_df.empty:
        seen_tickers = {t[0] for t in tech_targets}
        for _, row in entry_df.head(NN_HUNT_K).iterrows():
            if row['ticker'] not in seen_tickers:
                hunt_targets.append((row['ticker'], row['name'], '🎯狙い目'))

    ai_targets = tech_targets + hunt_targets
    print(f'\nStage 3 開始: テクニカル上位 {len(tech_targets)} + '
          f'狙い目 {len(hunt_targets)} = 計 {len(ai_targets)} 銘柄\n')
    print('  ※ 入力: 対数収益率   ※ 選択基準: MAPE÷(方向精度+ε) の最小化\n')

    nn_results = []
    for j, (tkr, nm, src_label) in enumerate(ai_targets, 1):
        print(f'  [{j}/{len(ai_targets)}] [{src_label}] {tkr} ({nm}) を訓練中...',
              flush=True)
        r = predict_with_best_model(
            tkr, nm, NN_TRAIN_YEARS, NN_LOOK_BACK,
            NN_FORECAST_DAYS, NN_EPOCHS, NN_BATCH)
        if r is None:
            print('    [スキップ] データ不足または取得失敗')
            continue
        r['source'] = src_label
        nn_results.append(r)

        lines = []
        for mn, v in r['per_model'].items():
            if v.get('mape') is not None:
                lines.append(f"{mn}: MAPE={v['mape']:.2f}% DirAcc={v['dir_acc']:.1%}")
            else:
                lines.append(f"{mn}: エラー")
        print('    ' + '  |  '.join(lines))
        print(f"    ⭐ 最適モデル: {r['best_model']}  "
              f"MAPE:{r['best_mape']:.2f}%  方向精度:{r['best_dir_acc']:.1%}")
        print(f"    現在 {r['current_price']:.2f}  →  {NN_FORECAST_DAYS}日後予測 "
              f"{r['forecast_end_price']:.2f}  ({r['forecast_return']:+.2f}%)  "
              f"AI信頼スコア: {r['ai_score']:+.1f}\n")

    stage3_elapsed = time.time() - stage3_start
    print(f'Stage 3 完了 （{stage3_elapsed:.1f} 秒）')
else:
    nn_results = []
    print('Stage 3 はスキップされました（RUN_NN_PREDICTION = False）')


In [ ]:
# ===== AI予測結果の可視化・複合スコアランキング =====
if RUN_NN_PREDICTION and nn_results:

    # ---- 複合スコアの計算 ----
    # テクニカルスコア（Stage 2）を 0〜1 に正規化して AI スコアと合算
    tech_scores  = ranking_df.set_index('ticker')['スコア'].to_dict()
    max_tech     = ranking_df['スコア'].max() + 1e-8

    for r in nn_results:
        ts = tech_scores.get(r['ticker'], 0)
        ts_norm = ts / max_tech * 100          # 0〜100 に正規化
        r['tech_score_raw']  = ts
        r['tech_score_norm'] = ts_norm
        # 複合スコア: テクニカル 60% + AI信頼スコア 40%
        r['combined_score']  = 0.6 * ts_norm + 0.4 * np.clip(r['ai_score'], -50, 50)
        # シグナル収束判定
        tech_up = ts >= 40                     # テクニカルが一定以上
        ai_up   = r['forecast_return'] > 0     # AI が上昇予測
        r['signal'] = ('🟢 両シグナル↑' if tech_up and ai_up
                       else ('🟡 テクニカル↑/AI↓' if tech_up
                             else ('🟠 AI↑/テクニカル低' if ai_up
                                   else '🔴 両シグナル↓')))

    nn_results_sorted = sorted(nn_results, key=lambda x: x['combined_score'], reverse=True)

    # ---- 複合ランキング表示 ----
    print('=' * 75)
    print('🏆 複合スコアランキング  （テクニカル 60% ＋ AI 40%）')
    print('=' * 75)
    print(f"{'順位':<4} {'経路':<6} {'銘柄':^18} {'複合':>6} {'テク':>6} {'AIスコア':>8} "
          f"{'AI予測%':>8} {'MAPE':>6} {'方向精度':>8} {'シグナル'}")
    print('-' * 82)
    for k, r in enumerate(nn_results_sorted, 1):
        src_lbl = r.get('source', 'テク上位')[:5]
        print(f"{k:<4} {src_lbl:<6} {r['ticker']:>8} {r['name'][:8]:<8} "
              f"{r['combined_score']:>6.1f} {r['tech_score_norm']:>6.1f} "
              f"{r['ai_score']:>+8.1f} {r['forecast_return']:>+8.2f}% "
              f"{r['best_mape']:>5.2f}% {r['best_dir_acc']:>8.1%} "
              f"  {r['signal']}")
    print()
    print('  🟢 両シグナル↑ : テクニカル・AI ともに上昇シグナル → 最も強い推奨')
    print('  🟡 テクニカル↑ : 過去の株価推移は良好だが AI は慎重')
    print('  🟠 AI↑のみ    : AI が上昇を見込むがテクニカルは低評価')
    print('  🔴 両シグナル↓ : テクニカル・AI ともに弱い → 見送り推奨')

    # ---- CSV 出力（拡張ランキング）----
    nn_df = pd.DataFrame([
        {
            'ticker': r['ticker'], 'name': r['name'],
            'AI最適モデル':          r['best_model'],
            'AI_MAPE%':              round(r['best_mape'], 2),
            '方向精度':              round(r['best_dir_acc'] * 100, 1),
            '現在価格':              round(r['current_price'], 2),
            f'{NN_FORECAST_DAYS}日後予測': round(r['forecast_end_price'], 2),
            'AI予測リターン%':       round(r['forecast_return'], 2),
            'AI信頼スコア':          round(r['ai_score'], 1),
            '複合スコア':            round(r['combined_score'], 1),
            'シグナル':              r['signal'],
            'AI分析経路':            r.get('source', 'テクニカル上位'),
        }
        for r in nn_results_sorted
    ])
    augmented_ranking = ranking_df.merge(nn_df, on=['ticker', 'name'], how='left')

    aug_filename = f"recommend_with_ai_{datetime.today().strftime('%Y%m%d')}.csv"
    try:
        augmented_ranking.to_csv(aug_filename, encoding='utf-8-sig')
        print(f'\n✓ AI予測込みランキングを保存: {aug_filename}')
        try:
            from google.colab import files as colab_files
            colab_files.download(aug_filename)
        except Exception:
            pass
    except Exception as e:
        print(f'⚠️ CSV保存に失敗: {e}')

    # ---- 可視化 1: 銘柄ごとの予測チャート ----
    n = len(nn_results_sorted)
    fig, axes = plt.subplots(n, 1, figsize=(14, 4.2 * n))
    if n == 1:
        axes = [axes]

    for ax, r in zip(axes, nn_results_sorted):
        recent = min(180, len(r['history_dates']))
        ax.plot(r['history_dates'][-recent:], r['history_prices'][-recent:],
                label='実績株価', color='#1565C0', linewidth=2)
        if len(r['test_dates']) > 0 and len(r['test_pred']) > 0:
            ax.plot(r['test_dates'], r['test_pred'],
                    label=f"{r['best_model']} テスト予測",
                    color='#FF8F00', linewidth=1.4, linestyle='--', alpha=0.85)
        ax.plot(r['forecast_dates'], r['forecast_prices'],
                label=f"{NN_FORECAST_DAYS}日先予測 ({r['forecast_return']:+.1f}%)",
                color='#C62828', linewidth=2.0, marker='o', markersize=3)
        ax.axvline(x=r['history_dates'][-1], color='#2E7D32',
                   linestyle=':', linewidth=1.2, alpha=0.7)
        ymax = ax.get_ylim()[1]
        ax.text(r['history_dates'][-1], ymax * 0.98, ' 現在',
                color='#2E7D32', fontsize=9, va='top')
        ax.set_title(
            f"{r.get('source','テク')} | {r['ticker']} ({r['name']})  "
            f"最適:{r['best_model']}  MAPE:{r['best_mape']:.2f}%  "
            f"方向精度:{r['best_dir_acc']:.1%}  {r['signal']}",
            fontsize=11)
        ax.set_ylabel('価格')
        ax.legend(loc='upper left', fontsize=9)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    ai_chart_file = 'ai_forecast.png'
    plt.savefig(ai_chart_file, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'✓ 予測チャートを保存: {ai_chart_file}')

    # ---- 可視化 2: 複合スコアと各シグナルの比較バーチャート ----
    fig2, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(14, max(3, n * 0.9 + 1)))

    labels   = [f"{r['ticker']}\n{r['name']}" for r in nn_results_sorted]
    y_pos    = np.arange(len(nn_results_sorted))
    bar_h    = 0.3

    tech_vals = [r['tech_score_norm'] for r in nn_results_sorted]
    ai_vals   = [np.clip(r['ai_score'], -50, 50) for r in nn_results_sorted]
    comb_vals = [r['combined_score']  for r in nn_results_sorted]

    # 左パネル: テクニカル vs AI 信頼スコアの比較
    ax_left.barh(y_pos + bar_h, tech_vals, bar_h, label='テクニカルスコア', color='#42A5F5')
    ax_left.barh(y_pos,          ai_vals,   bar_h, label='AI信頼スコア',
                 color=[('#26A69A' if v >= 0 else '#EF5350') for v in ai_vals])
    ax_left.axvline(0, color='black', linewidth=0.8)
    ax_left.set_yticks(y_pos + bar_h / 2)
    ax_left.set_yticklabels(labels, fontsize=9)
    ax_left.set_xlabel('スコア')
    ax_left.set_title('テクニカル vs AI シグナル', fontsize=11)
    ax_left.legend(fontsize=9)
    ax_left.grid(axis='x', alpha=0.3)
    ax_left.invert_yaxis()

    # 右パネル: 複合スコアランキング
    colors_comb = [('#1565C0' if v >= 40 else '#90A4AE') for v in comb_vals]
    hbars = ax_right.barh(labels, comb_vals, color=colors_comb, height=0.5)
    for bar, r in zip(hbars, nn_results_sorted):
        ax_right.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                      f"{r['combined_score']:.1f}  {r['signal']}",
                      va='center', fontsize=9)
    ax_right.set_xlabel('複合スコア（テクニカル 60% ＋ AI 40%）')
    ax_right.set_title('最終複合スコアランキング', fontsize=11)
    ax_right.set_xlim(0, max(comb_vals) * 1.35 if comb_vals else 10)
    ax_right.grid(axis='x', alpha=0.3)
    ax_right.invert_yaxis()

    plt.tight_layout()
    plt.savefig('ai_combined_ranking.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('✓ 複合ランキングチャートを保存: ai_combined_ranking.png')

    # ---- 可視化 3: モデル別 MAPE + 方向精度ヒートマップ ----
    model_names = list(MODEL_BUILDERS.keys())
    mape_mat  = np.full((len(nn_results_sorted), len(model_names)), np.nan)
    dir_mat   = np.full((len(nn_results_sorted), len(model_names)), np.nan)
    for i, r in enumerate(nn_results_sorted):
        for j, mn in enumerate(model_names):
            v = r['per_model'].get(mn, {})
            if v.get('mape') is not None:
                mape_mat[i, j] = v['mape']
                dir_mat[i,  j] = v['dir_acc'] * 100

    fig3, (ax_m, ax_d) = plt.subplots(1, 2, figsize=(12, max(3, n * 0.8 + 1)))
    for ax_h, mat, cmap_h, title_h, fmt_h in [
        (ax_m, mape_mat, 'RdYlGn_r', 'MAPE (%) — 低いほど良', '{:.1f}%'),
        (ax_d, dir_mat,  'RdYlGn',   '方向精度 (%) — 高いほど良', '{:.0f}%'),
    ]:
        masked = np.ma.masked_invalid(mat)
        im = ax_h.imshow(masked, cmap=cmap_h, aspect='auto')
        ax_h.set_xticks(range(len(model_names)))
        ax_h.set_xticklabels(model_names, fontsize=9)
        ax_h.set_yticks(range(len(nn_results_sorted)))
        ax_h.set_yticklabels([r['ticker'] for r in nn_results_sorted], fontsize=9)
        ax_h.set_title(title_h, fontsize=11)
        plt.colorbar(im, ax=ax_h, shrink=0.8)
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                if not np.isnan(mat[i, j]):
                    ax_h.text(j, i, fmt_h.format(mat[i, j]),
                              ha='center', va='center', fontsize=9, color='black')
        # 最適モデルに ⭐
        for i, r in enumerate(nn_results_sorted):
            if r['best_model'] in model_names:
                j = model_names.index(r['best_model'])
                ax_h.text(j, i - 0.3, '⭐', ha='center', va='bottom', fontsize=11)

    plt.tight_layout()
    plt.savefig('ai_model_heatmap.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('✓ モデル精度ヒートマップを保存: ai_model_heatmap.png')

    print('\n⚠️ 免責事項: AIによる予測は過去パターンに基づくものであり、将来の株価を保証しません。')
    print('   投資判断はご自身の責任で行ってください。')

elif RUN_NN_PREDICTION:
    print('⚠️ AI予測結果がありません（全銘柄でデータ不足）')
